# Encoder detectors for HALT

Runs two encoders on all five conditions and merges them with the linear
baselines into one comparison.

**Why two.** DeBERTa-v3-base caps at 512 tokens. Reports average ~1,500 tokens
and the judge that produced the labels saw ~1,125 (3,000 characters of report
plus 1,500 of reference). At 512 the detector sees *less than the judge did*,
so its numbers are a lower bound. ModernBERT handles 8,192 tokens, so running
it at 1,280 gives the detector exactly the judge's view. Reporting both
separates "the detector is weak" from "the detector was starved of input".

Runtimes on an A100, all five conditions:

| Run | Setting | Time |
|---|---|---|
| A DeBERTa-v3-base | 512 tokens, bs 16, 2 epochs | ~2.5 h |
| B ModernBERT-base | 1,280 tokens, bs 8, 2 epochs | ~4.5 h |

If you have already run A, skip to Part B. Colab disconnects on idle, so keep
the tab active.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CANDIDATES = [
    '/content/drive/Shareddrives/RESEARCH/2026/PROMPTS/RESULTS',
    '/content/drive/Shareddrives/RESEARCH/Research/2026/PROMPTS/RESULTS',
]

def ok(p):
    return os.path.isdir(os.path.join(p, 'EVALUATION-RESULT'))

RESULTS_ROOT = next((p for p in CANDIDATES if ok(p)), None)
if RESULTS_ROOT is None:
    raise SystemExit('Set RESULTS_ROOT by hand.')

OUT = f'{RESULTS_ROOT}/ANALYSIS-OUTPUT-RESULT/iclr_benchmark'
os.environ['HALLUBENCH_OUT'] = OUT
LABELS = f'{OUT}/benchmark_labels.csv.gz'

print('OUT    =', OUT)
print('labels =', 'found' if os.path.exists(LABELS) else 'MISSING')

In [ ]:
!pip -q install "transformers>=4.48" accelerate sentencepiece
import torch, transformers
print('transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — Runtime > Change runtime type > GPU')

## Write the training script

One shared encoder, seven binary heads (H1–H6 plus ANY), fed
`[report] [SEP] [reference]`. Class weights match the balanced logistic
baselines so the comparison is like-for-like. Warns if the chosen model's
context is shorter than `--max_len`.

In [ ]:
%%writefile /content/06_encoder_detector.py
#!/usr/bin/env python3
"""
Step 6 - fine-tuned encoder detector. RUN THIS ON A GPU (Colab A100 is enough).

Addresses the strongest objection to the paper: that the transfer collapse is
an artifact of linear detectors rather than a property of the task. Fine-tunes
one encoder per split with six per-type heads plus an ANY head, on
[report] [SEP] [reference], and evaluates on the standard, leave-one-source-out
(all three folds), and held-out-strategy splits.

Requires network access to download model weights, so it cannot run in the
offline analysis container.

    pip install "transformers>=4.48" torch scikit-learn accelerate
    python3 06_encoder_detector.py --model answerdotai/ModernBERT-base

CONTEXT LENGTH MATTERS HERE. Reports average ~1,500 tokens. The judge that
produced the labels saw 3,000 characters of report (~750 tokens) plus 1,500 of
reference (~375), so --max_len 1280 gives the detector exactly the judge's
view. At 512 the detector sees less than the judge did and the comparison is
unfair to it. ModernBERT handles 8,192 tokens, so 1,280 costs nothing.

Writes: encoder_results.csv  (same schema as baseline_results.csv, so the
        existing report and figure code consumes it unchanged)

Runtime guide, A100, ModernBERT-base, max_len 1280, bs 8, 2 epochs:
    standard split          ~60 min
    3 LOSO folds            ~150 min
    held-out strategy       ~60 min
Roughly 4.5 h in total. For the validation pass use
    --model roberta-base --max_len 512 --bs 16 --epochs 1
which takes about 20 minutes and only checks that the loop runs.
"""
import argparse, os
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

ap = argparse.ArgumentParser()
ap.add_argument('--model', default='answerdotai/ModernBERT-base')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/encoder_results.csv')
ap.add_argument('--max_len', type=int, default=1280,
                help='1280 matches what the judge saw; raise it to test whether '
                     'the detector benefits from more than the judge had')
ap.add_argument('--epochs', type=int, default=2)
ap.add_argument('--bs', type=int, default=8,
                help='lower than usual because of the long context')
ap.add_argument('--lr', type=float, default=2e-5)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

torch.manual_seed(args.seed); np.random.seed(args.seed)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if dev == 'cpu':
    print('WARNING: no GPU visible. This will take many hours.')

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')
tok = AutoTokenizer.from_pretrained(args.model)
_cap = getattr(tok, 'model_max_length', 512)
if _cap and _cap < args.max_len and _cap < 100000:
    print(f'WARNING: {args.model} caps at {_cap} tokens but --max_len is '
          f'{args.max_len}. Reports will be cut below what the judge saw. '
          f'Use a long-context model (ModernBERT, Longformer) or lower '
          f'--max_len and say so in the paper.')


class Reports(Dataset):
    """Report and its reference annotation as a sentence pair."""
    def __init__(self, frame):
        self.a = frame.model_output.tolist()
        self.b = frame.ground_truth.tolist()
        self.y = frame[TARGETS].to_numpy(dtype='float32')

    def __len__(self):
        return len(self.a)

    def __getitem__(self, i):
        enc = tok(self.a[i], self.b[i], truncation=True, max_length=args.max_len,
                  padding='max_length', return_tensors='pt')
        return ({k: v.squeeze(0) for k, v in enc.items()},
                torch.tensor(self.y[i]))


class MultiHead(torch.nn.Module):
    """One shared encoder, seven independent binary heads."""
    def __init__(self, name, n=len(TARGETS)):
        super().__init__()
        # force fp32: some checkpoints (DeBERTa-v3) declare a fp16 dtype in
        # their config, which makes GradScaler refuse to unscale gradients
        self.enc = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        d = self.enc.config.hidden_size
        self.drop = torch.nn.Dropout(0.1)
        self.heads = torch.nn.Linear(d, n)

    def forward(self, **kw):
        h = self.enc(**kw).last_hidden_state[:, 0]     # [CLS]
        return self.heads(self.drop(h))


def run(train_mask, test_mask, tag):
    tr, te = df[train_mask].reset_index(drop=True), df[test_mask].reset_index(drop=True)
    print(f'\n== {tag}: train {len(tr):,}  test {len(te):,}', flush=True)

    model = MultiHead(args.model).to(dev)
    dl_tr = DataLoader(Reports(tr), batch_size=args.bs, shuffle=True, num_workers=2)
    dl_te = DataLoader(Reports(te), batch_size=args.bs * 2, num_workers=2)

    # class weights per head, matching the balanced logistic baselines
    pos = tr[TARGETS].mean().to_numpy()
    w = torch.tensor(((1 - pos) / np.clip(pos, 1e-6, None)).astype('float32')).to(dev)
    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=w)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr)
    steps = len(dl_tr) * args.epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)

    # bf16 where the GPU supports it (A100 and newer): same dynamic range as
    # fp32, so no loss scaling is needed and GradScaler is skipped entirely.
    use_bf16 = dev == 'cuda' and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda' and not use_bf16))
    print(f'   precision: {"bf16" if use_bf16 else ("fp16+scaler" if dev=="cuda" else "fp32")}',
          flush=True)

    model.train()
    for ep in range(args.epochs):
        for i, (x, y) in enumerate(dl_tr):
            x = {k: v.to(dev) for k, v in x.items()}; y = y.to(dev)
            opt.zero_grad()
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                loss = lossf(model(**x), y)
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            sch.step()
            if i % 100 == 0:
                print(f'   ep{ep} step {i}/{len(dl_tr)} loss {loss.item():.4f}', flush=True)

    model.eval(); P = []
    with torch.no_grad():
        for x, _ in dl_te:
            x = {k: v.to(dev) for k, v in x.items()}
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                P.append(torch.sigmoid(model(**x)).float().cpu().numpy())
    P = np.vstack(P)

    rows = []
    for j, t in enumerate(TARGETS):
        y = te[t].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({'features': 'encoder', 'split': tag, 'target': t,
                     'auc': roc_auc_score(y, P[:, j]),
                     'f1': f1_score(y, (P[:, j] >= 0.5).astype(int)),
                     'pos_rate_test': float(y.mean())})
    del model; torch.cuda.empty_cache()
    return rows


out = []
out += run(df.split_random == 'train', df.split_random == 'test', 'random')
for held in ['Claude', 'GPT', 'Gemini']:
    out += run(df.model != held, df.model == held, f'heldout_model_{held}')
out += run(df.split_heldout_technique == 'train',
           df.split_heldout_technique == 'test', 'heldout_technique')

res = pd.DataFrame(out)
res.to_csv(args.out, index=False)
print('\n' + res.pivot_table(index='split', columns='target',
                             values='auc')[TARGETS].round(3).to_markdown())
print('\nwrote ->', args.out)


---
## Part A — DeBERTa-v3-base at 512 tokens

The short-context run. Its numbers are a **lower bound**: at 512 tokens the
detector sees roughly the first third of each report, less than the judge saw.

Skip this cell if `encoder_results_deberta512.csv` already exists.

In [ ]:
!python3 /content/06_encoder_detector.py \
    --model microsoft/deberta-v3-base --max_len 512 --bs 16 --epochs 2 \
    --labels "$LABELS" --out "$OUT/encoder_results_deberta512.csv" 

---
## Part B — ModernBERT-base at 1,280 tokens

The judge-matched run. 1,280 tokens is chosen deliberately: it is what the
judge saw, no more and no less.

In [ ]:
!python3 /content/06_encoder_detector.py \
    --model answerdotai/ModernBERT-base --max_len 1280 --bs 8 --epochs 2 \
    --labels "$LABELS" --out "$OUT/encoder_results_modernbert1280.csv" 

### Optional: does more context than the judge help?

One fold at 2,048 tokens. If the detector improves when it sees *more* than the
judge did, that is evidence the labels themselves are truncation-limited, which
connects to the analysis in Section 4.5. About 50 minutes.

In [ ]:
!python3 /content/06_encoder_detector.py \
    --model answerdotai/ModernBERT-base --max_len 2048 --bs 4 --epochs 2 \
    --labels "$LABELS" --out "$OUT/encoder_results_modernbert2048.csv" 

---
## Combined view

Merges both encoders with the linear baselines. This is the table the paper
needs: whether the degradation under source shift is a property of the task or
of weak detectors, and whether it is uniform across hallucination types.

In [ ]:
import numpy as np, pandas as pd, os

H = ['H1','H2','H3','H4','H5','H6']; T = H + ['any_hallucination']
AX = {'fabrication': ['H1','H5','H6'], 'omission': ['H3'], 'distortion': ['H2','H4']}

SOURCES = [
    ('baseline_results.csv',                 None),            # linear, split col already set
    ('encoder_results_deberta512.csv',       'deberta-512'),
    ('encoder_results_modernbert1280.csv',   'modernbert-1280'),
    ('encoder_results_modernbert2048.csv',   'modernbert-2048'),
]

frames = []
for fname, label in SOURCES:
    path = f'{OUT}/{fname}'
    if not os.path.exists(path):
        print('missing (skipped):', fname); continue
    d = pd.read_csv(path)
    if label:
        d['features'] = label
    frames.append(d)
    print('loaded:', fname, f'({len(d)} rows)')

allr = pd.concat(frames, ignore_index=True)
# collapse the three LOSO folds into one condition
allr['condition'] = allr.split.str.replace(r'heldout_model_.*', 'heldout_model',
                                           regex=True)

piv = allr.pivot_table(index=['features','condition'], columns='target',
                       values='auc')[T]
for a, cs in AX.items():
    piv[a] = piv[cs].mean(axis=1)
piv['macro'] = piv[H].mean(axis=1)
piv = piv.round(3)
piv.to_csv(f'{OUT}/all_detectors_summary.csv')
print()
print(piv.to_markdown())

### The comparison that decides the paper

Degradation from the standard split to leave-one-source-out, per axis, per
detector. If fabrication loses more than omission and distortion in every row,
the finding is robust across detector families. If the ANY column collapses for
some detectors and not others, that needs saying explicitly.

In [ ]:
rows = []
for feat in piv.index.get_level_values(0).unique():
    try:
        r = piv.loc[(feat, 'random')]
        l = piv.loc[(feat, 'heldout_model')]
    except KeyError:
        continue
    rows.append({'detector': feat,
                 'fabrication': round(l.fabrication - r.fabrication, 3),
                 'omission':    round(l.omission - r.omission, 3),
                 'distortion':  round(l.distortion - r.distortion, 3),
                 'macro':       round(l.macro - r.macro, 3),
                 'ANY':         round(l.any_hallucination - r.any_hallucination, 3),
                 'ANY (LOSO)':  round(l.any_hallucination, 3)})

deg = pd.DataFrame(rows).set_index('detector')
deg.to_csv(f'{OUT}/degradation_summary.csv')
print('Degradation, standard split -> leave-one-source-out\n')
print(deg.to_markdown())

print('\nDoes fabrication degrade most, in every detector?')
for d, r in deg.iterrows():
    worst = min([('fabrication', r.fabrication), ('omission', r.omission),
                 ('distortion', r.distortion)], key=lambda x: x[1])[0]
    ratio = r.fabrication / min(r.omission, r.distortion) if min(r.omission, r.distortion) < 0 else float('nan')
    print(f'  {d:18s} worst axis = {worst:12s} fabrication/least = {ratio:.1f}x')

### Per-fold detail

The fold spread matters: if held-out Claude behaves very differently from
held-out Gemini, that supports the claim that transfer depends on whether a
similar failure profile appears in training.

In [ ]:
per_fold = allr[allr.split.str.startswith('heldout_model_')]
if len(per_fold):
    p = per_fold.pivot_table(index=['features','split'], columns='target',
                             values='auc')[T].round(3)
    p.to_csv(f'{OUT}/per_fold_detail.csv')
    print(p.to_markdown())
else:
    print('no per-fold rows found')

### Send back

Paste the three tables above, and share:

- `all_detectors_summary.csv`
- `degradation_summary.csv`
- `per_fold_detail.csv`
- `encoder_results_deberta512.csv`, `encoder_results_modernbert1280.csv`

These become rows in Tables 3 and 4, bars in Figure 3, and they determine how
Sections 4.3 and 4.4 are written.